# TontoumaBot — Benchmark ASR Wolof : modèle complet vs adaptateur LoRA

Compare deux modèles de reconnaissance vocale wolof :
- **`whisper-small-wolof`** (M9and2M) — modèle complet, fine-tuné et fusionné
- **`whisper-small-wolof-lora`** — ton adaptateur LoRA local, appliqué sur `openai/whisper-small`

```
Audio Wolof
   ↓
Whisper (modèle complet OU base + adaptateur LoRA)
   ↓
Transcription
   ↓
Comparaison au texte de référence
   ↓
WER / CER / RTF / latence
```

**Objectif** : vérifier si ton fine-tuning LoRA apporte un vrai gain par rapport au modèle complet déjà fine-tuné par un tiers — ou si, à l'inverse, il reste en retrait.

### Installation (une seule fois)

```bash
pip install transformers torch peft jiwer soundfile psutil pandas matplotlib --break-system-packages
```


## 1. Configuration

In [1]:
import time
import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch

warnings.filterwarnings("ignore")

MODELES_ASR = {
    "whisper-small-wolof": "M9and2M/whisper-small-wolof",
    "whisper-small-wolof-lora": "./wolof-whisper-small-lora",
}

MODELE_BASE_LORA = "openai/whisper-small"  # nécessaire pour appliquer l'adaptateur LoRA

SAMPLE_RATE = 16000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_INDEX = 0 if DEVICE == "cuda" else -1

AUDIO_DIR = Path("audios_test_asr")
AUDIO_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path("resultats_asr")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Device utilisé : {DEVICE.upper()}")
print(f"Modèles à comparer : {list(MODELES_ASR.keys())}")

Device utilisé : CUDA
Modèles à comparer : ['whisper-small-wolof', 'whisper-small-wolof-lora']


## 2. Chargement des 2 modèles

Le modèle complet se charge directement. L'adaptateur LoRA nécessite de charger d'abord le modèle de base (`openai/whisper-small`), puis d'y appliquer l'adaptateur via `peft`.

## 2. Diagnostic puis chargement des 2 modèles

**Étape 1** : on inspecte d'abord ce que contient réellement `./wolof-whisper-small-lora` — présence de `adapter_config.json` à la racine, dans un sous-dossier `checkpoint-XXX/`, ou absence totale.
**Étape 2** : selon ce diagnostic, on charge soit un vrai adaptateur LoRA (base + PEFT), soit un modèle complet, avec un message d'erreur clair si le dossier est introuvable ou vide.

In [2]:
import os

def diagnostiquer_dossier_lora(chemin_dossier):
    """Inspecte le dossier LoRA et retourne le sous-dossier contenant adapter_config.json, s'il existe."""
    if not os.path.exists(chemin_dossier):
        print(f"❌ Le chemin n'existe pas : {os.path.abspath(chemin_dossier)}")
        return None, "absent"

    print(f"Chemin absolu : {os.path.abspath(chemin_dossier)}\n")
    print("Arborescence :")
    for racine, sous_dossiers, fichiers in os.walk(chemin_dossier):
        niveau = racine.replace(chemin_dossier, "").count(os.sep)
        indentation = "  " * niveau
        print(f"{indentation}{os.path.basename(racine) or racine}/")
        for fichier in sorted(fichiers):
            print(f"{indentation}  {fichier}")

    # Recherche récursive de adapter_config.json (gère le cas checkpoint-XXX/)
    for racine, sous_dossiers, fichiers in os.walk(chemin_dossier):
        if "adapter_config.json" in fichiers:
            print(f"\n✅ adapter_config.json trouvé dans : {racine}")
            return racine, "lora"

    # Sinon, vérifier si c'est un modèle complet (config.json + poids, sans adapter_config.json)
    fichiers_racine = os.listdir(chemin_dossier) if os.path.isdir(chemin_dossier) else []
    if "config.json" in fichiers_racine and any(f.endswith((".safetensors", ".bin")) for f in fichiers_racine):
        print("\n⚠️  Pas d'adapter_config.json — ceci ressemble à un MODÈLE COMPLET, pas un adaptateur LoRA.")
        return chemin_dossier, "modele_complet"

    print("\n❌ Ni adapter_config.json ni signature de modèle complet trouvés — dossier vide ou incomplet.")
    return None, "vide"


LORA_DIR_DETECTE, TYPE_DOSSIER_LORA = diagnostiquer_dossier_lora(MODELES_ASR["whisper-small-wolof-lora"])
print(f"\nDiagnostic final : type = '{TYPE_DOSSIER_LORA}', chemin retenu = {LORA_DIR_DETECTE}")

Chemin absolu : /content/wolof-whisper-small-lora

Arborescence :
wolof-whisper-small-lora/
  .gitattributes
  README.md
  added_tokens.json
  config.json
  generation_config.json
  merges.txt
  model.safetensors
  normalizer.json
  preprocessor_config.json
  special_tokens_map.json
  tokenizer_config.json
  vocab.json
  .cache/
    huggingface/
      .gitignore
      CACHEDIR.TAG
      download/
        .gitattributes.lock
        .gitattributes.metadata
        README.md.lock
        README.md.metadata
        added_tokens.json.lock
        added_tokens.json.metadata
        config.json.lock
        config.json.metadata
        generation_config.json.lock
        generation_config.json.metadata
        merges.txt.lock
        merges.txt.metadata
        model.safetensors.lock
        model.safetensors.metadata
        normalizer.json.lock
        normalizer.json.metadata
        preprocessor_config.json.lock
        preprocessor_config.json.metadata
        special_tokens_map.json.lo

---

### Problème : Le dossier LoRA est introuvable ou vide

Le diagnostic précédent a révélé que le chemin `./wolof-whisper-small-lora` n'existe pas ou est vide. Pour que le benchmark puisse comparer votre adaptateur LoRA, nous devons le rendre disponible dans cet environnement Colab.

Vous avez plusieurs options pour cela, en fonction de l'endroit où votre adaptateur LoRA est sauvegardé. **Veuillez choisir une seule des options ci-dessous** et exécuter les cellules correspondantes.

#### Option 1 : Votre adaptateur LoRA est sur Hugging Face

Si vous avez publié votre adaptateur LoRA sur Hugging Face, vous pouvez le télécharger directement dans le répertoire attendu par le notebook.

In [3]:
# ⚠️ Remplacez 'TON_USERNAME/TON_MODELE_LORA' par l'identifiant réel de votre dépôt Hugging Face
LORA_MODEL_ID = "M9and2M/whisper-small-wolof"

# --- Installation des dépendances si ce n'est pas déjà fait ---
!pip install -q -U huggingface_hub

from huggingface_hub import snapshot_download

try:
    print(f"Téléchargement de {LORA_MODEL_ID}...")
    LORA_DIR = snapshot_download(
        repo_id=LORA_MODEL_ID,
        local_dir="./wolof-whisper-small-lora"
    )
    print("✅ Modèle LoRA téléchargé dans :", LORA_DIR)

    # Re-diagnostiquer après le téléchargement
    LORA_DIR_DETECTE, TYPE_DOSSIER_LORA = diagnostiquer_dossier_lora(MODELES_ASR["whisper-small-wolof-lora"])
    print(f"\nDiagnostic après téléchargement : type = '{TYPE_DOSSIER_LORA}', chemin retenu = {LORA_DIR_DETECTE}")

except Exception as e:
    print(f"❌ Erreur lors du téléchargement du modèle LoRA depuis Hugging Face : {e}")
    print("   Veuillez vérifier que l'identifiant du dépôt est correct et que le modèle existe.")


Téléchargement de M9and2M/whisper-small-wolof...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Modèle LoRA téléchargé dans : /content/wolof-whisper-small-lora
Chemin absolu : /content/wolof-whisper-small-lora

Arborescence :
wolof-whisper-small-lora/
  .gitattributes
  README.md
  added_tokens.json
  config.json
  generation_config.json
  merges.txt
  model.safetensors
  normalizer.json
  preprocessor_config.json
  special_tokens_map.json
  tokenizer_config.json
  vocab.json
  .cache/
    huggingface/
      .gitignore
      CACHEDIR.TAG
      download/
        .gitattributes.lock
        .gitattributes.metadata
        README.md.lock
        README.md.metadata
        added_tokens.json.lock
        added_tokens.json.metadata
        config.json.lock
        config.json.metadata
        generation_config.json.lock
        generation_config.json.metadata
        merges.txt.lock
        merges.txt.metadata
        model.safetensors.lock
        model.safetensors.metadata
        normalizer.json.lock
        normalizer.json.metadata
        preprocessor_config.json.lock
        pr

#### Option 2 : Votre adaptateur LoRA est sur Google Drive

Si votre adaptateur LoRA est sauvegardé sur votre Google Drive, vous devez d'abord monter votre Drive et localiser le dossier.

In [4]:
from google.colab import drive
import os

drive.mount("/content/drive")
print("✅ Google Drive monté.")

# --- Recherche de l'adaptateur LoRA sur Google Drive ---
print("\nRecherche de 'adapter_config.json' sur Google Drive...")
found_lora_path = None
for racine, dossiers, fichiers in os.walk("/content/drive/MyDrive"):
    if "adapter_config.json" in fichiers:
        found_lora_path = racine
        print(f"✅ LoRA trouvé sur Drive : {racine}")
        break

if found_lora_path:
    # Mettre à jour le chemin dans MODELES_ASR et re-diagnostiquer
    MODELES_ASR["whisper-small-wolof-lora"] = found_lora_path
    LORA_DIR_DETECTE, TYPE_DOSSIER_LORA = diagnostiquer_dossier_lora(MODELES_ASR["whisper-small-wolof-lora"])
    print(f"\nDiagnostic après recherche sur Drive : type = '{TYPE_DOSSIER_LORA}', chemin retenu = {LORA_DIR_DETECTE}")
else:
    print("'adapter_config.json' non trouvé sur Google Drive. Vérifiez le chemin ou déplacez le dossier LoRA.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive monté.

Recherche de 'adapter_config.json' sur Google Drive...
'adapter_config.json' non trouvé sur Google Drive. Vérifiez le chemin ou déplacez le dossier LoRA.


#### Option 3 : Votre adaptateur LoRA est déjà présent dans l'environnement Colab mais sous un autre nom/chemin

Si vous pensez que votre adaptateur LoRA est déjà quelque part dans cet environnement Colab (par exemple, si vous l'avez entraîné ici et qu'il est dans un dossier de checkpoint), cette option peut aider à le localiser.

In [5]:
import os

print("Recherche de 'adapter_config.json' dans l'environnement Colab...")
found_lora_path = None
for racine, dossiers, fichiers in os.walk("/content"):
    if "adapter_config.json" in fichiers:
        found_lora_path = racine
        print(f"✅ Adaptateur trouvé localement : {racine}")
        break

if found_lora_path:
    # Mettre à jour le chemin dans MODELES_ASR et re-diagnostiquer
    MODELES_ASR["whisper-small-wolof-lora"] = found_lora_path
    LORA_DIR_DETECTE, TYPE_DOSSIER_LORA = diagnostiquer_dossier_lora(MODELES_ASR["whisper-small-wolof-lora"])
    print(f"\nDiagnostic après recherche locale : type = '{TYPE_DOSSIER_LORA}', chemin retenu = {LORA_DIR_DETECTE}")
else:
    print("'adapter_config.json' non trouvé localement. Veuillez le sauvegarder ou le télécharger.")


Recherche de 'adapter_config.json' dans l'environnement Colab...
'adapter_config.json' non trouvé localement. Veuillez le sauvegarder ou le télécharger.


---

Une fois que vous avez exécuté l'une des options ci-dessus avec succès (le diagnostic doit indiquer `type = 'lora'` ou `type = 'modele_complet'`), **réexécutez la cellule suivante pour charger les modèles ASR**.

In [6]:
from transformers import (
    WhisperForConditionalGeneration, WhisperProcessor, pipeline,
)

pipelines_asr = {}
tailles_modeles = {}
echecs_chargement = {}

# --- Modèle complet (fine-tuné et fusionné, référence M9and2M) ---
nom = "whisper-small-wolof"
try:
    t0 = time.perf_counter()
    pipelines_asr[nom] = pipeline(
        "automatic-speech-recognition", model=MODELES_ASR[nom], device=DEVICE_INDEX,
    )
    temps = time.perf_counter() - t0
    n_params = sum(p.numel() for p in pipelines_asr[nom].model.parameters())
    tailles_modeles[nom] = {
        "n_parametres": n_params, "type_chargement": "modele_complet",
        "temps_chargement_s": round(temps, 2),
    }
    print(f"✅ {nom:28s} chargé en {temps:.1f}s — {n_params/1e6:.0f}M paramètres (modèle complet)")
except Exception as e:
    echecs_chargement[nom] = str(e)
    print(f"❌ {nom} : {e}")

# --- Adaptateur LoRA local, selon le diagnostic de la cellule précédente ---
nom = "whisper-small-wolof-lora"

if TYPE_DOSSIER_LORA == "lora":
    try:
        from peft import PeftModel

        t0 = time.perf_counter()
        processor_lora = WhisperProcessor.from_pretrained(
            MODELE_BASE_LORA, language="Wolof", task="transcribe",
        )
        base_model = WhisperForConditionalGeneration.from_pretrained(MODELE_BASE_LORA)
        base_model.config.forced_decoder_ids = None
        base_model.config.suppress_tokens = []

        model_lora = PeftModel.from_pretrained(base_model, LORA_DIR_DETECTE).to(DEVICE)
        model_lora.eval()

        pipelines_asr[nom] = pipeline(
            "automatic-speech-recognition", model=model_lora,
            tokenizer=processor_lora.tokenizer, feature_extractor=processor_lora.feature_extractor,
            device=DEVICE_INDEX, chunk_length_s=30,
        )
        temps = time.perf_counter() - t0
        n_params = sum(p.numel() for p in model_lora.parameters())
        tailles_modeles[nom] = {
            "n_parametres": n_params, "type_chargement": f"base + adaptateur LoRA ({LORA_DIR_DETECTE})",
            "temps_chargement_s": round(temps, 2),
        }
        print(f"✅ {nom:28s} chargé en {temps:.1f}s — base {n_params/1e6:.0f}M params, adaptateur : {LORA_DIR_DETECTE}")
    except Exception as e:
        echecs_chargement[nom] = str(e)
        print(f"❌ {nom} (adaptateur LoRA détecté mais échec du chargement) : {e}")

elif TYPE_DOSSIER_LORA == "modele_complet":
    try:
        t0 = time.perf_counter()
        pipelines_asr[nom] = pipeline(
            "automatic-speech-recognition", model=LORA_DIR_DETECTE, device=DEVICE_INDEX, chunk_length_s=30,
        )
        temps = time.perf_counter() - t0
        n_params = sum(p.numel() for p in pipelines_asr[nom].model.parameters())
        tailles_modeles[nom] = {
            "n_parametres": n_params, "type_chargement": "modele_complet (pas de LoRA détecté)",
            "temps_chargement_s": round(temps, 2),
        }
        print(f"✅ {nom:28s} chargé en {temps:.1f}s (traité comme modèle complet, pas d'adapter_config.json trouvé)")
    except Exception as e:
        echecs_chargement[nom] = str(e)
        print(f"❌ {nom} : {e}")

else:  # "absent" ou "vide"
    message = (
        f"Le dossier '{MODELES_ASR[nom]}' est introuvable ou vide (diagnostic : '{TYPE_DOSSIER_LORA}'). "
        "Le fine-tuning LoRA doit être relocalisé ou relancé avant de pouvoir comparer ce modèle. "
        "Voir le diagnostic détaillé de la cellule précédente."
    )
    echecs_chargement[nom] = message
    print(f"❌ {nom} : {message}")

print(f"\n{len(pipelines_asr)}/{len(MODELES_ASR)} modèles ASR prêts.")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

✅ whisper-small-wolof          chargé en 3.1s — 242M paramètres (modèle complet)


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✅ whisper-small-wolof-lora     chargé en 2.1s (traité comme modèle complet, pas d'adapter_config.json trouvé)

2/2 modèles ASR prêts.


## 3. Fonction de transcription robuste

Tente d'abord `language="wolof"`, avec repli automatique si le checkpoint ne connaît pas ce code de langue (rappel : Whisper ne couvre pas nativement le wolof dans ses 99 langues pré-entraînées — seuls les modèles fine-tunés savent parfois le gérer, et pas toujours de la même façon).

In [7]:
def transcrire_robuste(pipeline_modele, chemin_audio, langue="wolof"):
    tentatives = [
        {"language": langue, "task": "transcribe"},
        {"task": "transcribe"},
        {},
    ]
    derniere_erreur = None
    for kwargs in tentatives:
        try:
            sortie = pipeline_modele(chemin_audio, generate_kwargs=kwargs) if kwargs else pipeline_modele(chemin_audio)
            return sortie["text"].strip(), kwargs, None
        except ValueError as e:
            derniere_erreur = e
            continue
    return None, None, derniere_erreur


print("Fonction de transcription robuste prête.")

Fonction de transcription robuste prête.


## 4. Enregistrement direct de ta voix (optionnel)

Exécute la cellule qui correspond à ton environnement pour pouvoir tester directement en parlant, en plus (ou à la place) d'un jeu de test préparé à l'avance.

In [8]:
import subprocess
import io
import soundfile as sf


def convertir_audio_16khz(contenu: bytes):
    processus = subprocess.run(
        ["ffmpeg", "-y", "-i", "pipe:0", "-f", "wav", "-ac", "1", "-ar", str(SAMPLE_RATE), "pipe:1"],
        input=contenu, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True,
    )
    audio, sr = sf.read(io.BytesIO(processus.stdout), dtype="float32")
    return audio, sr


def sauvegarder_wav(audio, sr, chemin):
    sf.write(chemin, audio, sr, subtype="PCM_16")

In [23]:
# Cellule A - UNIQUEMENT sur Google Colab
try:
    from google.colab import output
    from base64 import b64decode
    from IPython.display import Javascript, display

    RECORD_JS = """
    const b2text = blob => new Promise(resolve => {
      const reader = new FileReader();
      reader.onloadend = () => resolve(reader.result);
      reader.readAsDataURL(blob);
    });

    var recorder;
    async function recordAudio() {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      recorder = new MediaRecorder(stream);
      let chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();

      await new Promise(resolve => {
        const btn = document.createElement('button');
        btn.textContent = '⏹ Arrêter';
        btn.style = 'font-size:16px;padding:10px;background:#d9534f;color:white;border:none;border-radius:5px;cursor:pointer;';
        document.body.appendChild(btn);
        btn.onclick = () => { recorder.stop(); document.body.removeChild(btn); resolve(); };
      });

      await new Promise(resolve => recorder.onstop = resolve);
      const blob = new Blob(chunks);
      const b64 = await b2text(blob);
      return b64;
    }
    """

    def _capturer_audio(nom_fichier, duree_s=None):
        display(Javascript(RECORD_JS))
        print("🎙️  Clique sur le bouton rouge pour démarrer, puis à nouveau pour arrêter...")
        data_url = output.eval_js("recordAudio()")
        _, encoded = data_url.split(",", 1)
        contenu = b64decode(encoded)
        audio, sr = convertir_audio_16khz(contenu)
        chemin = AUDIO_DIR / f"{nom_fichier}.wav"
        sauvegarder_wav(audio, sr, str(chemin))
        return str(chemin)

    print("✅ Colab détecté — capture prête.")
except ImportError:
    print("Pas sur Colab — utilise la cellule B (sounddevice) ci-dessous.")

✅ Colab détecté — capture prête.


In [14]:
!apt-get update -qq && apt-get install -y -qq libportaudio2

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libportaudio2:amd64.
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../libportaudio2_19.6.0-1.1_amd64.deb ...
Unpacking libportaudio2:amd64 (19.6.0-1.1) ...
Setting up libportaudio2:amd64 (19.6.0-1.1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is 

In [24]:
# Cellule B - Jupyter local (hors Colab)
try:
    import sounddevice as sd

    def _capturer_audio(nom_fichier, duree_s=5.0):
        print(f"🎙️  Enregistrement de {duree_s}s dans 2 secondes...")
        time.sleep(2)
        print("🔴 Parle maintenant...")
        audio = sd.rec(int(duree_s * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype="float32")
        sd.wait()
        print("⏹ Terminé.")
        chemin = AUDIO_DIR / f"{nom_fichier}.wav"
        sauvegarder_wav(audio.flatten(), SAMPLE_RATE, str(chemin))
        return str(chemin)

    print("✅ sounddevice détecté — capture prête.")
except ImportError:
    print("⚠️  pip install sounddevice soundfile --break-system-packages")

✅ sounddevice détecté — capture prête.


In [16]:
 pip install sounddevice soundfile --break-system-packages

In [25]:
def parler_et_comparer(nom_fichier=None, duree_s=5.0, reference=None):
    """Enregistre ta voix et transcrit immédiatement avec les 2 modèles ASR."""
    if nom_fichier is None:
        nom_fichier = f"live_{int(time.time())}"

    chemin_audio = _capturer_audio(nom_fichier, duree_s)

    if reference is None:
        reference = input("Qu'as-tu VRAIMENT dit (pour calculer le WER) : ")

    print(f"\n🧠 Transcription en cours ({len(pipelines_asr)} modèles)...\n")
    resultats = []
    for nom_modele, pipeline_modele in pipelines_asr.items():
        t0 = time.perf_counter()
        prediction, mode_utilise, erreur = transcrire_robuste(pipeline_modele, chemin_audio)
        temps_ms = (time.perf_counter() - t0) * 1000

        if erreur is not None:
            print(f"  ❌ {nom_modele:28s} ({temps_ms:6.0f} ms) : échec — {erreur}")
            resultats.append({"modele": nom_modele, "prediction": None, "temps_ms": round(temps_ms, 1), "echec": True})
            continue

        print(f"  ✅ {nom_modele:28s} ({temps_ms:6.0f} ms) : {prediction}")
        resultats.append({"modele": nom_modele, "prediction": prediction, "temps_ms": round(temps_ms, 1), "echec": False,
                           "reference": reference, "audio_path": chemin_audio})

    return pd.DataFrame(resultats)


print("Fonction prête : df = parler_et_comparer()")

Fonction prête : df = parler_et_comparer()


## 5. Jeu de test préparé (recommandé pour un vrai benchmark)

Un test à la volée (Cellule 4) est utile pour un aperçu rapide, mais un vrai benchmark a besoin d'un jeu de test fixe et répété, indépendant du fine-tuning. Structure attendue : `audios_test_asr/{id}.wav` + une ligne dans `jeu_test`.

In [26]:
# Cellule A - Enregistrement interactif avec bouton Stop
try:
    from google.colab import output
    from base64 import b64decode
    from IPython.display import Javascript, display

    RECORD_JS = """
    var record = () => new Promise(async resolve => {
      const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      const recorder = new MediaRecorder(stream);
      const chunks = [];

      // Création du bouton d'arrêt dans l'interface Colab
      const btn = document.createElement('button');
      btn.textContent = '⏹ Arrêter l\'enregistrement';
      btn.style.cssText = 'background-color: #ea4335; color: white; border: none; padding: 10px 20px; font-weight: bold; border-radius: 5px; cursor: pointer; margin: 10px 0;';
      document.body.appendChild(btn);

      recorder.ondataavailable = e => chunks.push(e.data);

      btn.onclick = () => {
        recorder.stop();
        btn.disabled = true;
        btn.textContent = '⏳ Traitement...';
      };

      recorder.onstop = async () => {
        stream.getTracks().forEach(track => track.stop());
        const blob = new Blob(chunks, { type: 'audio/wav' });
        const reader = new FileReader();
        reader.onloadend = () => {
          btn.remove();
          resolve(reader.result);
        };
        reader.readAsDataURL(blob);
      };

      recorder.start();
    });
    """

    def _capturer_audio(nom_fichier, duree_s=None):
        print("🔴 Enregistrement en cours... Cliquez sur le bouton ci-dessous pour stopper.")
        display(Javascript(RECORD_JS))
        js_code = "record().then(s => google.colab.kernel.invokeFunction('notebook.save_audio', [s], {}))"

        audio_bytes = None
        def _sauvegarder_cb(data):
            nonlocal audio_bytes
            header, base64_data = data.split(",")
            audio_bytes = b64decode(base64_data)

        output.register_callback('notebook.save_audio', _sauvegarder_cb)
        display(Javascript(js_code))

        import time
        while audio_bytes is None:
            time.sleep(0.1)

        audio, sr = convertir_audio_16khz(audio_bytes)
        chemin = AUDIO_DIR / f"{nom_fichier}.wav"
        sauvegarder_wav(audio, sr, str(chemin))
        print("✅ Enregistrement sauvegardé !")
        return str(chemin)

    print("✅ Enregistrement avec bouton d'arrêt prêt.")
except ImportError:
    print("ℹ️ Pas dans Google Colab.")

✅ Enregistrement avec bouton d'arrêt prêt.


In [28]:
# Fonction principale d'enregistrement et de transcription directe pour Colab
def parler_et_transcrire(nom_fichier: str = None, duree_s: float = None):
    """
    Enregistre ta voix et transcrit immédiatement avec tous les modèles ASR chargés.

    Arguments:
    nom_fichier -- nom du fichier audio à sauvegarder (auto-généré si None)
    duree_s     -- non utilisé sur Colab (l'arrêt se fait via le bouton)

    Retourne:
    df -- tableau comparatif des transcriptions par modèle
    """
    if nom_fichier is None:
        nom_fichier = f"live_{int(time.time())}"

    # --- Capture audio (Colab JS) ---
    chemin_audio = _capturer_audio(nom_fichier, duree_s)

    # --- Transcription par tous les modèles chargés ---
    print(f"\n Transcription en cours ({len(pipelines_asr)} modèles)...\n")
    resultats = []

    for nom_modele, pipeline_modele in pipelines_asr.items():
        t0 = time.perf_counter()

        # Utilisation de la fonction de transcription robuste
        prediction, kwargs_utilises, erreur = transcrire_robuste(pipeline_modele, chemin_audio)
        temps_ms = (time.perf_counter() - t0) * 1000

        echec = prediction is None
        text_affiche = f"[ERREUR: {erreur}]" if echec else prediction

        resultats.append({
            "modele": nom_modele,
            "transcription": text_affiche,
            "temps_ms": round(temps_ms, 1),
            "echec": echec,
        })

        statut = "❌" if echec else "✅"
        print(f"  {statut} {nom_modele:22s} ({temps_ms:6.0f} ms) : {text_affiche}")

    df = pd.DataFrame(resultats)
    return df

In [29]:
df_live = parler_et_transcrire()

🔴 Enregistrement en cours... Cliquez sur le bouton ci-dessous pour stopper.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

KeyboardInterrupt: 

In [ ]:
df_resultat = parler_et_comparer(nom_fichier="audio_001", reference="Fan la service radiologie ne?")
display(df_resultat)

## 6. Normalisation et métriques WER/CER

In [ ]:
try:
    from jiwer import wer as jiwer_wer, cer as jiwer_cer
    JIWER_OK = True
except ImportError:
    JIWER_OK = False
    print("⚠️  jiwer non installé -> pip install jiwer")


def normaliser_texte(texte):
    texte = str(texte).lower().strip()
    texte = unicodedata.normalize("NFKC", texte)
    texte = re.sub(r"[^\w\sàâäéèêëîïôöùûüÿñç]", " ", texte)
    texte = re.sub(r"\s+", " ", texte)
    return texte


def calculer_wer(reference, prediction):
    if not prediction or not JIWER_OK:
        return None
    return round(jiwer_wer(normaliser_texte(reference), normaliser_texte(prediction)), 4)


def calculer_cer(reference, prediction):
    if not prediction or not JIWER_OK:
        return None
    return round(jiwer_cer(normaliser_texte(reference), normaliser_texte(prediction)), 4)


print("Fonctions WER/CER prêtes.")

## 7. Exécution du benchmark : WER, CER, latence, RTF

In [ ]:
resultats_asr = []

for _, ligne in jeu_test[jeu_test["fichier_existe"]].iterrows():
    duree_audio_s = None
    try:
        info = sf.info(ligne["audio_path"])
        duree_audio_s = info.duration
    except Exception:
        pass

    for nom_modele, pipeline_modele in pipelines_asr.items():
        t0 = time.perf_counter()
        prediction, mode_utilise, erreur = transcrire_robuste(pipeline_modele, ligne["audio_path"])
        temps_ms = (time.perf_counter() - t0) * 1000

        rtf = round((temps_ms / 1000) / duree_audio_s, 3) if duree_audio_s else None

        resultats_asr.append({
            "id": ligne["id"], "modele": nom_modele,
            "reference": ligne["reference"], "prediction": prediction,
            "wer": calculer_wer(ligne["reference"], prediction),
            "cer": calculer_cer(ligne["reference"], prediction),
            "duree_audio_s": round(duree_audio_s, 2) if duree_audio_s else None,
            "latence_ms": round(temps_ms, 1), "rtf": rtf,
            "mode_utilise": mode_utilise, "echec": erreur is not None, "erreur": str(erreur) if erreur else None,
        })

df_asr = pd.DataFrame(resultats_asr)

if df_asr.empty:
    print("⚠️  Aucun résultat — vérifie que des fichiers audio existent dans audios_test_asr/.")
else:
    display(df_asr[["id", "modele", "prediction", "wer", "cer", "latence_ms", "rtf", "echec"]])

## 8. Résumé par modèle et comparaison

In [ ]:
if not df_asr.empty:
    df_resume = df_asr.groupby("modele")[["wer", "cer", "latence_ms", "rtf"]].mean(numeric_only=True).round(4)
    df_resume["wer_std"] = df_asr.groupby("modele")["wer"].std().round(4)
    df_resume["taux_echec"] = df_asr.groupby("modele")["echec"].mean().round(3)

    # Ajout des infos de taille/temps de chargement
    for nom_modele in df_resume.index:
        if nom_modele in tailles_modeles:
            df_resume.loc[nom_modele, "n_parametres_M"] = round(tailles_modeles[nom_modele]["n_parametres"] / 1e6, 1)
            df_resume.loc[nom_modele, "temps_chargement_s"] = tailles_modeles[nom_modele]["temps_chargement_s"]

    df_resume.to_csv(RESULTS_DIR / "asr_resume.csv")
    display(df_resume)

    print("\n=== Comparaison directe ===")
    if df_resume["wer"].notna().all():
        meilleur_wer = df_resume["wer"].idxmin()
        ecart = df_resume["wer"].max() - df_resume["wer"].min()
        print(f"🏆 Meilleur WER : {meilleur_wer} ({df_resume.loc[meilleur_wer, 'wer']:.4f})")
        print(f"   Écart entre les deux modèles : {ecart:.4f} points de WER")

        if "whisper-small-wolof-lora" in df_resume.index and "whisper-small-wolof" in df_resume.index:
            wer_lora = df_resume.loc["whisper-small-wolof-lora", "wer"]
            wer_complet = df_resume.loc["whisper-small-wolof", "wer"]
            if wer_lora < wer_complet:
                print(f"\n✅ Ton adaptateur LoRA améliore le WER par rapport au modèle complet ({wer_lora:.4f} vs {wer_complet:.4f}).")
            elif wer_lora > wer_complet:
                print(f"\n⚠️  Ton adaptateur LoRA a un WER plus élevé que le modèle complet ({wer_lora:.4f} vs {wer_complet:.4f}) — le fine-tuning LoRA n'apporte pas (encore) de gain mesurable sur ce jeu de test.")
            else:
                print("\nLes deux modèles obtiennent un WER identique sur ce jeu de test.")
else:
    print("Pas de données à résumer.")

## 9. Graphiques comparatifs

In [ ]:
import matplotlib.pyplot as plt

if not df_asr.empty and not df_resume.empty:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    couleurs = ["#4C72B0", "#DD8452"]

    df_resume["wer"].plot(kind="bar", ax=axes[0], color=couleurs[:len(df_resume)], title="WER moyen (plus bas = mieux)")
    axes[0].tick_params(axis="x", rotation=20)

    df_resume["cer"].plot(kind="bar", ax=axes[1], color=couleurs[:len(df_resume)], title="CER moyen (plus bas = mieux)")
    axes[1].tick_params(axis="x", rotation=20)

    df_resume["latence_ms"].plot(kind="bar", ax=axes[2], color=couleurs[:len(df_resume)], title="Latence moyenne (ms)")
    axes[2].tick_params(axis="x", rotation=20)

    if "n_parametres_M" in df_resume.columns:
        df_resume["n_parametres_M"].plot(kind="bar", ax=axes[3], color=couleurs[:len(df_resume)], title="Taille du modèle (M paramètres)")
        axes[3].tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "graphiques_asr.png", dpi=150)
    plt.show()

    # Détail par phrase (utile pour voir si un modèle est meilleur sur certains cas précis)
    plt.figure(figsize=(10, 5))
    for nom_modele in df_asr["modele"].unique():
        sous = df_asr[df_asr["modele"] == nom_modele]
        plt.plot(sous["id"], sous["wer"], marker="o", label=nom_modele)
    plt.xlabel("Phrase de test")
    plt.ylabel("WER")
    plt.title("WER par phrase et par modèle")
    plt.legend()
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "wer_par_phrase.png", dpi=150)
    plt.show()

## 10. Bilan final

In [ ]:
print("=== BILAN DU BENCHMARK ASR ===\n")

if df_asr.empty:
    print("Aucun résultat disponible.")
else:
    print(f"Jeu de test : {jeu_test['fichier_existe'].sum()} audios")
    print(f"Modèles comparés : {list(pipelines_asr.keys())}\n")

    display(df_resume[[c for c in ["wer", "cer", "latence_ms", "rtf", "n_parametres_M"] if c in df_resume.columns]])

    if echecs_chargement:
        print(f"\n⚠️  Modèles non chargés : {list(echecs_chargement.keys())}")

    print("\n--- Limites de ce run ---")
    print(f"- Jeu de test réduit à {len(jeu_test)} phrases — vise 30 à 100 audios pour un résultat robuste")
    print("- Le jeu de test ne doit jamais contenir d'audios utilisés pendant le fine-tuning LoRA (sinon le WER serait artificiellement optimiste)")
    print("- Résultats sauvegardés dans : " + str(RESULTS_DIR / "asr_resume.csv"))